In [ ]:
# pip install sentence-transformers

In [1]:
from sentence_transformers import SentenceTransformer
from redis.commands.search.query import Query
from redis.commands.search.field import TextField, TagField, VectorField
from redis.commands.search.index_definition import IndexDefinition, IndexType
from redis.commands.json.path import Path

import numpy as np
import redis

ModuleNotFoundError: No module named 'redis.commands.search.index_definition'

In [2]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [3]:
r = redis.Redis(decode_responses=True)

Connect to Redis and delete any index previously created with the name vector_idx. 

(The dropindex() call throws an exception if the index doesn't already exist, which is why you need the try: except: block.)

In [8]:
try:
    r.ft("vector_idx").dropindex(True)
except redis.exceptions.ResponseError:
    pass

The schema in the example below specifies hash objects for storage and includes three fields: 
- the text content to index, 
- a tag field to represent the "genre" of the text, and 
- the embedding vector generated from the original text content. 

The embedding field specifies HNSW indexing, the L2 vector distance metric, Float32 values to represent the vector's components, and 384 dimensions, as required by the all-MiniLM-L6-v2 embedding model.

In [9]:
schema = (
    TextField("content"),
    TagField("genre"),
    VectorField("embedding", "HNSW", {
        "TYPE": "FLOAT32",
        "DIM": 384,
        "DISTANCE_METRIC":"L2"
    })
)

Explanation:

- r.ft("vector_idx"): Accesses the RediSearch index named "vector_idx" in your Redis instance.
- .create_index(schema, ...): Creates a new RediSearch index with the specified schema (fields and their types).
- schema: Defines which fields to index (e.g., text, tag, vector fields).
- definition=IndexDefinition(...): Specifies how and where to find the documents to index.
- prefix=["doc:"]: Only Redis keys starting with "doc:" will be indexed. This organizes your data and avoids indexing unrelated keys.
- index_type=IndexType.HASH: Tells RediSearch to index documents stored as Redis Hashes (field-value pairs).

In [10]:
r.ft("vector_idx").create_index(
    schema,
    definition=IndexDefinition(
        prefix=["doc:"], index_type=IndexType.HASH
    )
)

'OK'

#### Add data 
- When you add documents to Redis using hset() and the key starts with the prefix (e.g., doc:), RediSearch will automatically index those documents according to your schema.
- To store vector embeddings in a Redis Hash, use model.encode() from SentenceTransformer to generate the embedding for your text.
- Convert the embedding to a NumPy array of type float32 using .astype(np.float32).
- Use .tobytes() to encode the vector as a single binary string, which is the required format for vector fields in Redis Hashes.
For JSON documents (not Hashes), store the embedding as a list of floats instead of a binary string.

This ensures your data is indexed and searchable using vector queries in RediSearch.

In [12]:
content = "That is a very happy person"

r.hset("doc:0", mapping={
                    "content": content,
                    "genre": "persons",
                    "embedding": model.encode(content).astype(np.float32).tobytes(),
})

3

In [13]:
content = "That is a happy dog"

r.hset("doc:1", mapping={
                    "content": content,
                    "genre": "pets",
                    "embedding": model.encode(content).astype(np.float32).tobytes(),
                }
       )

content = "Today is a sunny day"

r.hset("doc:2", mapping={
                    "content": content,
                    "genre": "weather",
                    "embedding": model.encode(content).astype(np.float32).tobytes(),
                }
       )

3

In [14]:
q = Query(
    "*=>[KNN 3 @embedding $vec AS vector_distance]"
).return_field("vector_distance").return_field("content").dialect(2)

query_text = "That is a happy person"

res = r.ft("vector_idx").search(
    q, query_params={
        "vec": model.encode(query_text).astype(np.float32).tobytes()
    }
)

print(res)

Result{3 total, docs: [Document {'id': 'doc:0', 'payload': None, 'vector_distance': '0.114169985056', 'content': 'That is a very happy person'}, Document {'id': 'doc:1', 'payload': None, 'vector_distance': '0.610845267773', 'content': 'That is a happy dog'}, Document {'id': 'doc:2', 'payload': None, 'vector_distance': '1.48624777794', 'content': 'Today is a sunny day'}]}


More examples and details can be found in the RediSearch documentation: 


In [ ]:
try:
    r.ft("vector_json_idx").dropindex(True)
except redis.exceptions.ResponseError:
    pass

schema = (
    TextField("$.content", as_name="content"),
    TagField("$.genre", as_name="genre"),
    VectorField(
        "$.embedding", "HNSW", {
            "TYPE": "FLOAT32",
            "DIM": 384,
            "DISTANCE_METRIC": "L2"
        },
        as_name="embedding"
    )
)

r.ft("vector_json_idx").create_index(
    schema,
    definition=IndexDefinition(
        prefix=["jdoc:"], index_type=IndexType.JSON
    )
)

content = "That is a very happy person"

r.json().set("jdoc:0", Path.root_path(), {
    "content": content,
    "genre": "persons",
    "embedding": model.encode(content).astype(np.float32).tolist(),
})

content = "That is a happy dog"

r.json().set("jdoc:1", Path.root_path(), {
    "content": content,
    "genre": "pets",
    "embedding": model.encode(content).astype(np.float32).tolist(),
})

content = "Today is a sunny day"

r.json().set("jdoc:2", Path.root_path(), {
    "content": content,
    "genre": "weather",
    "embedding": model.encode(content).astype(np.float32).tolist(),
})

q = Query(
    "*=>[KNN 3 @embedding $vec AS vector_distance]"
).return_field("vector_distance").return_field("content").dialect(2)

query_text = "That is a happy person"

res = r.ft("vector_json_idx").search(
    q, query_params={
        "vec": model.encode(query_text).astype(np.float32).tobytes()
    }
)

print(repr(res))
